# AI Virtual Assistant for Customer Service - LaunchPad

This notebook deploys the local-NIM LaunchPad lab path. It expects the LaunchPad-managed Jupyter container to have been prepared from a host terminal with:

```bash
bash launchpad/setup-managed-jupyter.sh
```

After that setup, Docker commands in this notebook and in Jupyter terminals use the LaunchPad host Docker daemon.

## What This Starts

- Nemotron 3 Nano local NIM on GPU 0
- Embedding NIM, reranking NIM, and GPU Milvus on GPU 1
- Prebuilt GHCR application containers for the agent, retrievers, analytics, API gateway, and UI
- A LaunchPad UI proxy on host port 3001

You need an NGC personal API key with access to NIM containers and model assets.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import time
from collections import deque

JUPYTER_ROOT = Path("/opt/nvidia/launchpad/jupyter-notebook")
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"
DOCKER_SOCKET = JUPYTER_ROOT / "docker.sock"
if "DOCKER_HOST" not in os.environ and DOCKER_SOCKET.exists():
    os.environ["DOCKER_HOST"] = f"unix://{DOCKER_SOCKET}"


def find_repo_root():
    candidates = [
        JUPYTER_ROOT / "ai-virtual-assistant",
        Path.cwd(),
        *Path.cwd().parents,
        Path.home() / "ai-virtual-assistant",
    ]
    for candidate in candidates:
        if (candidate / "deploy" / "compose" / "docker-compose.yaml").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find deploy/compose/docker-compose.yaml. Run launchpad/setup-managed-jupyter.sh from a host terminal first.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

ENV_FILE = REPO_ROOT / ".env.launchpad"
COMPOSE_FILES = [
    REPO_ROOT / "deploy" / "compose" / "docker-compose.yaml",
    REPO_ROOT / "deploy" / "compose" / "docker-compose.ghcr.yaml",
    REPO_ROOT / "launchpad" / "docker-compose.launchpad.yaml",
]
COMPOSE_ARGS = []
for compose_file in COMPOSE_FILES:
    if not compose_file.exists():
        raise FileNotFoundError(compose_file)
    COMPOSE_ARGS.extend(["-f", str(compose_file)])

LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
DEPLOY_LOG = LOG_DIR / "ai_virtual_assistant_launchpad.log"

print(f"Repository root: {REPO_ROOT}")
print(f"Environment file: {ENV_FILE}")
print("Compose files:")
for compose_file in COMPOSE_FILES:
    print(f"- {compose_file}")
print(f"DOCKER_HOST: {os.environ.get('DOCKER_HOST', '<default>')}")

## Configure The LaunchPad Environment

Paste an NGC personal API key when prompted. The notebook writes `.env.launchpad` from `launchpad/.env.example` and keeps the file local to this instance.

In [ ]:
NGC_API_KEY = os.environ.get("NGC_API_KEY", "")
if not NGC_API_KEY or NGC_API_KEY == "<paste-ngc-api-key>":
    NGC_API_KEY = getpass.getpass("Enter your NGC personal API key: ")

if not NGC_API_KEY:
    raise ValueError("NGC_API_KEY is required for local NIM containers.")

template = (REPO_ROOT / "launchpad" / ".env.example").read_text(encoding="utf-8")
env_text = template.replace("NGC_API_KEY=<paste-ngc-api-key>", f"NGC_API_KEY={NGC_API_KEY}")
ENV_FILE.write_text(env_text, encoding="utf-8")
ENV_FILE.chmod(0o600)
os.environ["NGC_API_KEY"] = NGC_API_KEY

nim_cache = Path("/home/nvidia/.cache/nim")

print(f"Wrote {ENV_FILE}")
print(f"NIM model cache on the LaunchPad host: {nim_cache}")

## Check Docker Access

If this cell fails, run `bash launchpad/setup-managed-jupyter.sh` from Code Server or WebSSH, then refresh this notebook.

In [ ]:
def docker_cmd(*args):
    return ["docker", *map(str, args)]


def run_capture(command, input_text=None):
    result = subprocess.run(command, input=input_text, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(map(str, command))}")
    return result.stdout.strip()


def run_logged(command, log_file, error_message):
    recent_lines = deque(maxlen=10)
    print(f"Running: {' '.join(map(str, command))}", flush=True)
    print(f"Streaming output to {log_file}", flush=True)
    last_progress = time.monotonic()

    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\n\n$ {' '.join(map(str, command))}\n")
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.strip()
            if stripped:
                recent_lines.append(stripped)
            now = time.monotonic()
            if now - last_progress >= 10:
                print(".", end="", flush=True)
                last_progress = now
        return_code = process.wait()

    print(" done", flush=True)
    if return_code != 0:
        print(error_message)
        print(f"Full output: {log_file}")
        print("Last output lines:")
        for line in recent_lines:
            print(line)
        raise RuntimeError(f"Command failed with exit code {return_code}.")


print(run_capture(docker_cmd("compose", "version")))
print(run_capture(docker_cmd("ps", "--format", "table {{.Names}}\t{{.Image}}\t{{.Status}}")))

## Authenticate To NGC

In [ ]:
login = subprocess.run(
    docker_cmd("login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"),
    input=NGC_API_KEY,
    text=True,
    capture_output=True,
)
if login.returncode != 0:
    print(login.stdout)
    print(login.stderr)
    raise RuntimeError("Docker login to nvcr.io failed. Confirm this is an NGC personal API key with NIM access.")

print("Docker is authenticated with nvcr.io.")

## Validate The Compose Configuration

In [ ]:
config = subprocess.run(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "config"),
    text=True,
    capture_output=True,
)
if config.returncode != 0:
    print(config.stdout)
    print(config.stderr)
    raise RuntimeError("Docker Compose config validation failed.")

rendered = config.stdout
assert "nvcr.io/nim/nvidia/nemotron-3-nano" in rendered
assert "nvidia/nemotron-3-nano-30b-a3b" in rendered
assert "milvusdb/milvus" in rendered and "gpu" in rendered
assert "18086" in rendered
assert "agent-frontend-proxy" in rendered

print("Docker Compose configuration validated for LaunchPad local NIMs and GPU Milvus.")

## Start The Stack

The first run can take a long time because the NIM containers download model assets into `/home/nvidia/.cache/nim`. Full output is written to `logs/ai_virtual_assistant_launchpad.log`.

In [ ]:
DEPLOY_LOG.write_text("", encoding="utf-8")

build_ui = os.environ.get("AIVA_BUILD_LAUNCHPAD_UI", "1").strip().lower() not in {"0", "false", "no"}
if build_ui:
    run_logged(["bash", "launchpad/build-launchpad-ui.sh"], DEPLOY_LOG, "LaunchPad UI image build failed.")

run_logged(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "pull", "--policy", "missing"),
    DEPLOY_LOG,
    "Docker Compose image pull failed.",
)

run_logged(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "up", "-d", "--no-build"),
    DEPLOY_LOG,
    "Docker Compose deployment failed.",
)

print("LaunchPad stack start requested.")
print(f"Full output: {DEPLOY_LOG}")

## Check Status

In [ ]:
ps = subprocess.run(
    docker_cmd("compose", "--env-file", ENV_FILE, *COMPOSE_ARGS, "--profile", "local-nim", "ps"),
    text=True,
    capture_output=True,
)
print(ps.stdout)
if ps.returncode != 0:
    print(ps.stderr)

print("Open the LaunchPad secure link for port 3001 after the services are healthy.")
print("For detailed logs, use a Jupyter terminal or Code Server terminal, for example:")
print("docker logs -f nemollm-inference-microservice")

## Prepare Sample Data

In [ ]:
run_logged(["bash", "data/download.sh", "data/list_manuals.txt"], DEPLOY_LOG, "Sample manual download failed.")
print("Manual PDFs are ready in data/manuals_pdf.")
print("Next, open ai-virtual-assistant/notebooks/ingest_data.ipynb and choose the AIVA LaunchPad kernel.")

## Ingest Data

Open and run:

```text
ai-virtual-assistant/notebooks/ingest_data.ipynb
```

Choose the `AIVA LaunchPad` kernel if prompted. The setup script configures that kernel to reach the host-published unstructured retriever on port `18086`.

After ingestion finishes, open the LaunchPad secure link for port `3001` and try the sample AI Virtual Assistant UI.